# ARISTA temporal gene and ligand–receptor patterns — CytoBridge API

**Objective.** Recompute the dense-time gene programs and ligand–receptor communication profiles used for Supplementary Figures S15–S17 using only public `CytoBridge` APIs.

## Plan

1. Simulate observed and intermediate states with the shared interpolation workflow.
2. Recompute attention-based communication matrices at every dense timepoint.
3. Inverse-project PCA states from the package-processed reference H5AD and cluster temporal gene programs.
4. Project communication onto ligand–receptor expression, cluster LR profiles, and export prototype/small-multiple panels.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path
import sys
import torch

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'downstream_helpers').exists():
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from downstream_helpers.arista_api import (
    AristaSpatiotemporalConfig,
    assert_package_only_runtime,
    run_arista_temporal_programs_api,
)
from downstream_helpers.runner import display_svg_outputs

DEVICE = os.environ.get('CYTOBRIDGE_DEVICE', 'cuda' if torch.cuda.is_available() else 'cpu')
SMOKE = os.environ.get('CYTOBRIDGE_SMOKE', '0') == '1'
MODEL_FORMAT = os.environ.get('ARISTA_MODEL_FORMAT', 'legacy')
ALIGNED_H5AD = os.environ.get('ARISTA_ALIGNED_H5AD') or None
MODEL_DIR = os.environ.get('ARISTA_MODEL_DIR') or None
REFERENCE_H5AD = os.environ.get('ARISTA_REFERENCE_H5AD') or ALIGNED_H5AD
LR_DATABASE = os.environ.get('ARISTA_LR_DATABASE')
PAPER_PARITY = os.environ.get('ARISTA_PAPER_PARITY', '0') == '1'
PCA_COMPONENTS_CSV = os.environ.get('ARISTA_PCA_COMPONENTS_CSV') or None
PCA_CENTER_CSV = os.environ.get('ARISTA_PCA_CENTER_CSV') or None
CLASSIFIER_CACHE_PATH = os.environ.get('ARISTA_CLASSIFIER_CACHE_PATH') or None
DEVICE, SMOKE, PAPER_PARITY, MODEL_FORMAT, REFERENCE_H5AD, LR_DATABASE

## Configuration

The prospective workflow uses a package-processed `ARISTA_REFERENCE_H5AD` retaining `varm['PCs']`. For the declared paper contract, set `ARISTA_PAPER_PARITY=1` and provide the archived PCA loading and center tables through `ARISTA_PCA_COMPONENTS_CSV` and `ARISTA_PCA_CENTER_CSV`; their paths and hashes are recorded in the manifest. This mode uses the historical first-symbol LR rule, Ward linkage, dendrogram ordering, 3,072 generated particles, and the legacy communication sampling contract. Smoke mode still uses only 16 particles and three timepoints to validate wiring and is never a scientific comparison.

In [ ]:
if MODEL_FORMAT == 'current' and (ALIGNED_H5AD is None or MODEL_DIR is None):
    raise ValueError('Current mode requires ARISTA_ALIGNED_H5AD and ARISTA_MODEL_DIR.')
if REFERENCE_H5AD is None:
    raise ValueError('Set ARISTA_REFERENCE_H5AD to the package-processed arista_aligned.h5ad.')
if LR_DATABASE is None:
    raise ValueError('Set ARISTA_LR_DATABASE to CellChatDB.ligrec.human.csv.')
if PAPER_PARITY and (PCA_COMPONENTS_CSV is None or PCA_CENTER_CSV is None):
    raise ValueError(
        'ARISTA_PAPER_PARITY=1 requires ARISTA_PCA_COMPONENTS_CSV and '
        'ARISTA_PCA_CENTER_CSV.'
    )

config_kwargs = {}
if SMOKE:
    config_kwargs.update(
        time_points=(0.0, 1.0),
        interp_time_points=(0.5,),
        plot_3d_time_points=(0.0, 0.5, 1.0),
        n_samples=16,
    )
elif PAPER_PARITY:
    config_kwargs.update(n_samples=3072)
config = AristaSpatiotemporalConfig(
    output_name='arista_temporal_gene_lr_patterns_api' + ('_smoke' if SMOKE else ''),
    aligned_h5ad=ALIGNED_H5AD,
    model_dir=MODEL_DIR,
    model_format=MODEL_FORMAT,
    skip_nonsplit_sde=PAPER_PARITY,
    classifier_cache_path=CLASSIFIER_CACHE_PATH,
    classifier_epochs=300 if PAPER_PARITY else 1000,
    classifier_knn_neighbors=10 if PAPER_PARITY else 1,
    spatial_warp_to_observed_piecewise=not PAPER_PARITY,
    random_seed=42,
    device=DEVICE,
    run_communication=True,
    run_3d=False,
    **config_kwargs,
)
config

In [ ]:
result = run_arista_temporal_programs_api(
    config,
    lr_database=LR_DATABASE,
    reference_h5ad=REFERENCE_H5AD,
    n_top_genes=25 if SMOKE else 250,
    n_gene_clusters=2,
    n_lr_clusters=2,
    pca_components_csv=PCA_COMPONENTS_CSV,
    pca_center_csv=PCA_CENTER_CSV,
    preferred_species_tag=None if PAPER_PARITY else 'hs',
    gene_profile_cluster_order='raw' if PAPER_PARITY else 'peak_time',
    lr_profile_linkage_method='ward' if PAPER_PARITY else 'average',
    lr_profile_cluster_order='dendrogram' if PAPER_PARITY else 'peak_time',
    communication_max_cells_per_timepoint=3072 if PAPER_PARITY else None,
    communication_rng_warmup_max_cells_per_timepoint=2500 if PAPER_PARITY else None,
)
assert_package_only_runtime()
result

## Results

The heatmap and gene prototypes correspond to the computational inputs for S15. LR prototypes and all-pair small multiples correspond to S16 and S17. The historical 68-pair input set is reproducible with `ARISTA_PAPER_PARITY=1`, but the 22/46 Ward split is a boundary-sensitive derived result: the frozen manuscript table is 22/46, the published checkpoint recomputation is 24/44, and the selected retrained model is 11/57. Compare continuous profiles and scores in addition to hard cluster labels.

In [ ]:
display_svg_outputs([
    result.gene_heatmap,
    result.gene_pattern_figure,
    result.lr_prototype_figure,
    result.lr_profiles_figure,
])
print(result.manifest_path.read_text(encoding='utf-8'))

## Next checks

- Confirm that all nine dense timepoints are present and that self-loops are removed.
- Compare final nonzero LR pair count and the two cluster sizes with the SI target (22 and 46).
- Run GO enrichment from the exported gene lists and report the database/version separately.